In [1]:
import torch  
from transformers import AutoModelForCausalLM, AutoTokenizer  

tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
model = AutoModelForCausalLM.from_pretrained("state-spaces/mamba-130m-hf", dtype=torch.float32, device_map="auto",)  
model.eval()

d:\Hu_Module\Master\Semester 4\Study Project\Linear attention state management\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.
Loading weights: 100%|██████████| 242/242 [00:14<00:00, 16.16it/s, Materializing param=backbone.norm_f.weight]                  


MambaForCausalLM(
  (backbone): MambaModel(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-23): 24 x MambaBlock(
        (norm): MambaRMSNorm(768, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (act): SiLUActivation()
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(768, eps=1e-05)
  )
  (lm_head): Linear(in_features=768, out_features=50280, bias=False)
)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
print(torch.__version__)
print(torch.version.cuda)

2.11.0+cu126
12.6


In [4]:
from transformers import MambaCache

def feed_synthetic_ssm_state(model, ssm_states, conv_states):
    cache = MambaCache(config=model.config, max_batch_size=1, device=model.device, dtype=model.dtype)
    cache.ssm_states = [s.detach().clone() for s in ssm_states]
    cache.conv_states = [s.detach().clone() for s in conv_states]
    return cache

In [5]:
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)



with torch.no_grad():
    outputs = model(**inputs, use_cache=True)
    original_cache = outputs.cache_params
print(outputs)
print("______________")
ssm_states = original_cache.ssm_states
conv_states = original_cache.conv_states
seq_offset = 0
cache = {"ssm_states": ssm_states, "conv_states": conv_states, "seq_offset": seq_offset}
torch.save(cache, "cache.pt")
loaded_cache = torch.load("cache.pt")
loaded_ssm_states = loaded_cache["ssm_states"]
loaded_conv_states = loaded_cache["conv_states"]
loaded_seq_offset = loaded_cache["seq_offset"]

synthetic_cache = feed_synthetic_ssm_state(model, loaded_ssm_states, loaded_conv_states)

prompt = "The capital of France is Paris. The capital of Germany is Berlin"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
print(synthetic_cache)
with torch.no_grad():
    cache_position = torch.arange(loaded_seq_offset,inputs["input_ids"].size(1)+loaded_seq_offset, device=device)
    outputs = model(
        **inputs,
        cache_params=synthetic_cache,
        cache_position=cache_position,
    )
print(outputs.cache_params)

MambaCausalLMOutput(loss=None, logits=tensor([[[ 34.6690,  21.0415,  31.9019,  ...,  20.9368,  20.7224,  21.0288],
         [-56.1207, -66.3254, -55.1947,  ..., -66.3670, -66.2041, -66.3695],
         [ 32.9502,  24.4136,  30.7899,  ...,  24.3877,  24.0616,  24.3999],
         [-54.3084, -69.5757, -53.5541,  ..., -69.6775, -69.2875, -69.5035],
         [ 39.0968,  26.5671,  38.9996,  ...,  26.5397,  26.2446,  26.7183]]],
       device='cuda:0'), cache_params=<transformers.models.mamba.modeling_mamba.MambaCache object at 0x00000171A6C26250>, hidden_states=None)
______________


In [6]:
print(original_cache.ssm_states[0].shape)
print(original_cache.conv_states[0].shape)
print(len(original_cache.ssm_states))
print(len(original_cache.conv_states))

torch.Size([1, 1536, 16])
torch.Size([1, 1536, 4])
24
24


In [7]:
import torch.nn as nn

n_layers = len(original_cache.ssm_states)
state_shape = original_cache.ssm_states[0].shape

flatten_dim = state_shape.numel()
latent_dim = 512

class StateAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat
    
    def fit(self, states, num_epochs=10, learning_rate=1e-3, device="cpu"):
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        for epoch in range(num_epochs):
            for state in states:
                optimizer.zero_grad()
                _, compressed_state = self(state)
                loss = criterion(compressed_state, state)
                print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")
                loss.backward()
                optimizer.step()

ae = StateAutoencoder(flatten_dim, latent_dim).to(device)
print(state_shape)

torch.Size([1, 1536, 16])


In [8]:
from datasets import load_dataset

def load_data(dataset_name, split="train"):
    dataset = load_dataset(dataset_name, split=split)
    return dataset

dataset = load_data("nvidia/Nemotron-RL-Instruction-Following-MultiTurnChat-v1")
data_sample = dataset[0]
print(data_sample.keys())

dict_keys(['uuid', 'task_id', 'agent_ref', 'responses_create_params', 'rubric', 'context', 'metadata', 'pass_rate', 'pass_rate_total', 'pass_rate_passed'])


In [9]:
def extract_sessions(dataset):
    sessions = []

    for sample in dataset:
        dialog = sample["responses_create_params"]["input"]
        sessions.append(dialog)

    return sessions
sessions = extract_sessions(dataset)
print(type(sessions[0]))

<class 'list'>


In [10]:
sessions[0][0].keys()

dict_keys(['role', 'content'])

In [11]:
def build_turns(session):
    turns = []
    instruction = ""
    user_chat = ""
    assistant_chat = ""
    turn_index = 0
    for turn in session:
        if turn["role"] == "system":
            instruction = "System: " + turn["content"]
        elif turn["role"] == "user":
            user_chat = "User: " + turn["content"]
        elif turn["role"] == "assistant":
            assistant_chat = "Assistant: " + turn["content"]
            turns.append((instruction, user_chat, assistant_chat))
            user_chat = ""
            assistant_chat = ""
    return turns

In [12]:
turns = build_turns(sessions[0])
print(turns[0][1:])

('User: As a bailiff, you must maintain order and precision in all responses. From now on, every answer you provide must include exactly 3 sentences only. Explain the standard procedure for securing a courtroom before a session begins. Check out the outcome under permitted boundaries and it should be factually correct.', 'Assistant: 1 - Before a session begins, the bailiff arrives early to perform a systematic sweep of the courtroom, verifying that all entry points are clear, furniture is properly arranged, and any evidence or exhibits are securely stored.\n \n 2 - The bailiff then engages the security team to lock exterior doors, activate surveillance systems, test audio‑visual equipment, and ensure that all electronic devices are either turned off or placed in a controlled zone.\n \n 3 - Finally, a detailed checklist is completed, the courtroom key is logged, and a brief walk‑through confirms that the environment meets the court’s safety and order standards.')


In [13]:
from benchmarks.perplexity_benchmark import evaluate, plot_perplexity_comparison, plot_latency_comparison_exp2, plot_memory_growth_exp2
